# 05. Score the 2026 season → `futures/futures_predictions.csv`

The last notebook in the pipeline and **the only one that writes the artifact the live page reads**.
It fits nothing. It loads the frozen model `04` saved, applies it to the published 2026 schedule, and
writes 32 rows.

**What it produces:** a projected win count and a win *distribution* for each team, from 20,000
simulated seasons.

**What it deliberately does not produce.** The scaffold for this notebook planned market columns
(`win_total_line`, `book`, `p_over`, `p_under`, `p_push`). Those are **dropped**. Two independent
reasons, either sufficient:

1. `data_audit.json` reads `tier_c_open: false`. PREREGISTRATION §7 gate C is shut, and a
   probability against a posted line is exactly what gate C guards. The scaffold text predates
   Amendment 1, which is what closed gate C permanently for an archive source.
2. There is no 2026 line to attach. The archive covers 2014-2022, 2024 and 2025. Emitting
   all-null market columns would be dead schema on the page.

**Claim licence carried in every row** (from `model_metadata.json`): *does not beat the archived
market consensus; BACKTESTED, NOT LIVE-VALIDATED.*

| Section | What it does |
|---|---|
| 1 | Parameters |
| 2 | Paths, artifacts, and the §7 gate |
| 3 | Load the frozen model, verify hash and feature order |
| 4 | Build the 2026 game design |
| 5 | Simulate 20,000 seasons |
| 6 | Assemble the artifact frame and scan it |
| 7 | Write the CSV and read it back |

## Section 1. Parameters

Papermill knobs only. Everything that defines the model (family, feature order, alpha, tau, the
fitted constants) is **read from the saved bundle**, never set here. `N_SIMS` and `SEED` default to
the values `04` recorded so a default run reproduces the smoke simulation it already ran.

In [ ]:
AUDIT_PATH = None      # None -> futures/artifacts/data_audit.json
PANEL_PATH = None      # None -> futures/data/team_season_panel.parquet
META_PATH  = None      # None -> futures/artifacts/model_metadata.json   (04)
VENUE_PATH = None      # None -> futures/data/season_schedule_context.parquet
SCHED_PATH = None      # None -> futures/data/schedules_snapshot.parquet
MODEL_PATH = None      # None -> futures/models/win_totals_model.pkl
OUT_PATH   = None      # None -> futures/futures_predictions.csv
N_SIMS     = 20000
SEED       = 20260802
WRITE_ARTIFACTS = True
RUN_TESTS  = True

Seven paths, two simulation controls, two switches. Note what is absent: no season, no model name, no
feature list. The predict season comes from the audit and the model identity comes from the pkl.

### Section 1 test

The point of the parameter block is what it *cannot* do. This asserts the simulation controls are
sane and that no name capable of redefining the model has leaked into the parameter namespace. It is the
same check `04` makes, because the failure mode is the same: a papermill flag quietly swapping the
model out from under the artifact.

In [ ]:
if RUN_TESTS:
    assert isinstance(SEED, int) and N_SIMS >= 10000
    for _n in ("MODEL_FAMILY", "TAU", "ALPHA", "FEATURES", "PREDICT_SEASON", "TIER_C_OPEN"):
        assert _n not in dir(), f"{_n} must be read from an artifact, never a parameter"
    print(f"✓ Section 1 tests passed | {N_SIMS:,} simulations, seed {SEED}, "
          f"model identity and predict season not injectable")

Passed. The notebook has no parameter that can change *what* is predicted, only how many times the
season is simulated.

## Section 2. Paths, artifacts, and the §7 gate

Resolve the repo root by landmark, load `data_audit.json` and `model_metadata.json`, and read the
gate state out of them. The gate is a real barrier: this notebook writes the page's artifact, so it
must refuse to run unless the audit says `GO`/`GO-TIER-B` **and** §7 gate A passed. Gate B is read
too, not as a permission but because its `False` has to be carried into the output.

In [ ]:
import hashlib
import json
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "app.py").exists() and (p / "futures").is_dir():
            return p
    raise RuntimeError(f"repo root not found above {start}")


REPO    = _find_repo_root(Path.cwd())
FUTURES = REPO / "futures"
ART     = FUTURES / "artifacts"
DATA    = FUTURES / "data"
MODELS  = FUTURES / "models"

AUDIT = Path(AUDIT_PATH) if AUDIT_PATH else ART / "data_audit.json"
PANEL = Path(PANEL_PATH) if PANEL_PATH else DATA / "team_season_panel.parquet"
META  = Path(META_PATH) if META_PATH else ART / "model_metadata.json"
VENUE = Path(VENUE_PATH) if VENUE_PATH else DATA / "season_schedule_context.parquet"
SCHED = Path(SCHED_PATH) if SCHED_PATH else DATA / "schedules_snapshot.parquet"
MODEL = Path(MODEL_PATH) if MODEL_PATH else MODELS / "win_totals_model.pkl"
OUT   = Path(OUT_PATH) if OUT_PATH else FUTURES / "futures_predictions.csv"


def _rel(p):
    p = Path(p)
    try:
        return p.resolve().relative_to(REPO).as_posix()
    except ValueError:
        return str(p.resolve())


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


audit = json.loads(AUDIT.read_text(encoding="utf-8"))
meta  = json.loads(META.read_text(encoding="utf-8"))

VERDICT        = audit["verdict"]
TIER_C_OPEN    = bool(audit["tier_c_open"])
PREDICT_SEASON = int(audit["predict_season"]["season"])
FEATURES       = list(meta["features"]["columns"])
TARGET         = audit["target"]["column"]
GATE_A         = bool(meta["evidence"]["gate_A_passed"])
GATE_B         = bool(meta["evidence"]["gate_B_passed"])
CLAIM_LABEL    = meta["claim_licence"]["required_label"]
MODEL_FAMILY   = meta["model"]["family"]

print(f"repo            : {REPO}")
print(f"audit verdict   : {VERDICT}   (gate C open: {TIER_C_OPEN})")
print(f"gate A / gate B : {GATE_A} / {GATE_B}")
print(f"predict season  : {PREDICT_SEASON}   schedule published: "
      f"{audit['predict_season']['schedule_published']}, results present: "
      f"{audit['predict_season']['results_present']}")
print(f"model           : {MODEL_FAMILY}, {len(FEATURES)} features")
print(f"claim licence   : {CLAIM_LABEL}")
print(f"writes          : {_rel(OUT)}")

The gate reads `GO-TIER-B` with gate C shut, gate A passed and gate B failed, exactly the state `04` froze.
The predict season is 2026 with a published schedule and **zero results present**, which is the
condition that makes this a forecast rather than a mid-season update.

`CLAIM_LABEL` is read from the metadata rather than typed here on purpose: the sentence that goes on
the page is the one `04` recorded, and if that changes this notebook changes with it.

### Section 2 test

The gate has to be able to stop the notebook, so this asserts each condition separately rather than
in one compound expression. A `NO-GO`, a re-opened gate C, a failed gate A, or a gate B that
somehow flipped to `True` must each halt here. The last one matters most: gate B passing would be a
much stronger claim than any evidence supports, and `05` is not the notebook allowed to discover it.

In [ ]:
if RUN_TESTS:
    assert VERDICT in ("GO", "GO-TIER-B"), f"audit verdict {VERDICT!r} does not permit publishing"
    assert TIER_C_OPEN is False
    assert GATE_A is True, "§7 gate A did not pass - nothing may be published"
    assert GATE_B is False, "gate B is recorded as passing - 05 is not where that is decided"
    assert audit["predict_season"]["schedule_published"] is True
    assert int(audit["predict_season"]["results_present"]) == 0, \
        "the predict season has results - this is not a preseason forecast"
    assert "does not beat" in CLAIM_LABEL and "BACKTESTED" in CLAIM_LABEL
    assert len(FEATURES) == 25 and "market_line" not in FEATURES
    for p in (PANEL, VENUE, SCHED, MODEL):
        assert p.exists(), f"missing input: {p}"
    print(f"✓ Section 2 tests passed | verdict {VERDICT}, gate A True / gate B False / "
          f"gate C shut, {PREDICT_SEASON} has a schedule and no results, all 4 inputs present")

Passed. Every input exists and the gate permits exactly one thing: a descriptive projection with a
distribution, carrying the "does not beat" label.

## Section 3. Load the frozen model

`04` saved a bundle, not a notebook state. This loads it and rebuilds the small dict shape
`m4_engine` expects (`imp`/`sc`/`model`/`sigma`/`hfa`/`tie_thr`), then verifies three things the
artifact depends on: the file on disk is byte-identical to the one `04` hashed, the feature order in
the bundle matches the recorded contract, and the engine module is the one the bundle names.

Nothing is fitted here. If this notebook could fit, the published model and the evaluated model
could differ.

In [ ]:
sys.path.insert(0, str(FUTURES / "season_team_totals"))
import m4_engine as eng
from tier_lock import TierCViolation, assert_no_tier_c

BUNDLE = joblib.load(MODEL)
MODEL_HASH = sha256_file(MODEL)

FIT = {"imp": BUNDLE["imputer"], "sc": BUNDLE["scaler"], "model": BUNDLE["ridge"],
       "sigma": BUNDLE["sigma"], "hfa": BUNDLE["hfa"], "tie_thr": BUNDLE["tie_threshold"]}
TAU = float(BUNDLE["tau"])

ALLOWED = {VERDICT}


def guard(obj, where):
    assert_no_tier_c(obj, where, allowed_literals=ALLOWED, tier_c_open=TIER_C_OPEN)


print(f"family          : {BUNDLE['model_family']}")
print(f"sha256          : {MODEL_HASH}")
print(f"matches 04      : {MODEL_HASH == meta['model']['sha256']}")
print(f"engine          : {BUNDLE['engine']}")
print(f"trained on      : {BUNDLE['train_seasons'][0]}-{BUNDLE['train_seasons'][-1]} "
      f"({len(BUNDLE['train_seasons'])} seasons, {BUNDLE['n_train_games']:,} games)")
print(f"alpha / tau     : {BUNDLE['alpha']:g} / {TAU:g}")
print(f"home field      : {FIT['hfa']:.4f} pts")
print(f"margin sigma    : {FIT['sigma']:.4f}")
print(f"tie threshold   : {FIT['tie_thr']:.6f}")

The bundle on disk hashes to the value `04` recorded, so the model being published is the model that
was evaluated. Its training window ends at 2025 and the predict season is 2026, so the model has never
seen a 2026 outcome. None exist yet.

`tau = 5` is the Amendment 3 shock, and it sits at the top of the frozen grid. `04` recorded that as
a limitation (`tau_at_grid_boundary`), not a result; nothing here re-selects it.

### Section 3 test

Three failure modes worth catching: a stale or swapped pkl (hash), a feature reorder (which would
silently feed the ridge the wrong columns, since the coefficients are positional), and a dead Tier-C
guard. The guard check is a **red control**: it hands the guard a phrase that must raise, so a
passing scan later means the scanner ran rather than that it was inert.

In [ ]:
if RUN_TESTS:
    assert MODEL_HASH == meta["model"]["sha256"], "pkl on disk differs from the one 04 recorded"
    assert BUNDLE["feature_cols"] == FEATURES, "feature order differs from the recorded contract"
    assert BUNDLE["model_family"] == MODEL_FAMILY == "M4-c"
    assert PREDICT_SEASON not in BUNDLE["train_seasons"], "the model was trained on the predict season"
    assert BUNDLE["ridge"].coef_.shape[0] == len(FEATURES) + 1, "coef count != features + home field"
    assert TAU in eng.TAU_GRID
    try:
        guard({"note": "a betting edge"}, "selftest")
        raise AssertionError("guard inactive")
    except TierCViolation:
        pass
    print(f"✓ Section 3 tests passed | pkl hash matches 04, {len(FEATURES)} features in "
          f"contract order, {BUNDLE['ridge'].coef_.shape[0]} coefficients, "
          f"{PREDICT_SEASON} absent from training, Tier-C guard live")

Passed. The loaded object is provably the evaluated model, and the guard demonstrably fires.

## Section 4. Build the 2026 game design

The model is game-level: it predicts a home margin from the *difference* between the two teams'
season features, plus a home-field column that is switched off for neutral-site and international
games (A2.5.6). So scoring 2026 means building one design row per scheduled game.

The schedule is filtered exactly as `04` filtered it (regular season only, same season window)
because an inconsistent filter is how the venue join stops being 1:1.

In [ ]:
panel = pd.read_parquet(PANEL)
venue = pd.read_parquet(VENUE)
sched = pd.read_parquet(SCHED)

SEASON_MIN = int(audit["outcomes"]["season_min"])
SEASON_MAX = max(int(audit["outcomes"]["season_max"]), PREDICT_SEASON)
sched = sched[(sched["game_type"] == "REG") & sched["season"].between(SEASON_MIN, SEASON_MAX)].copy()

FR = {"OAK": "LV", "SD": "LAC", "STL": "LA"}
sched["home_franchise"] = sched["home_team"].replace(FR)
sched["away_franchise"] = sched["away_team"].replace(FR)

venue["no_home_field"] = venue["explicit_neutral"] | venue["international_game"]
games = sched.merge(venue[["game_id", "no_home_field"]], on="game_id", how="inner")
games["hfa_mult"] = np.where(games["no_home_field"], 0.0, 1.0)

feat = panel.set_index(["season", "franchise"])[FEATURES]
g26, X26 = eng.game_design(games, feat, [PREDICT_SEASON], FEATURES, settled_only=False)

TEAMS = sorted(set(g26["home_franchise"]) | set(g26["away_franchise"]))
neutral = int((g26["hfa_mult"] == 0).sum())

print(f"{PREDICT_SEASON} schedule : {len(g26)} games, {len(TEAMS)} teams")
print(f"home field off  : {neutral} games (neutral or international)")
print(f"design matrix   : {X26.shape[0]} x {X26.shape[1]}")
print(f"feature nulls   : {int(X26.isna().sum().sum())} cells "
      f"(median-imputed by the fitted imputer, not here)")
print()
print(g26.loc[g26['hfa_mult'] == 0, ['game_id', 'home_franchise', 'away_franchise', 'location']]
      .to_string(index=False))

272 games, 32 teams, and the home-field column is switched off for the 9 games listed above. All 9
are 2026's international slate, carried through from the venue table `01` corrected under
Amendment 2.5 (Melbourne, Rio, Tottenham ×2, Wembley, Paris, Madrid, Munich, Mexico City).

**Note the eighth row.** `2026_05_PHI_JAX` has `location = Home`. It is Jacksonville's designated
home game, but it is played at Tottenham. Eight of the nine are flagged `explicit_neutral`; this one
is not, and is caught only by `international_game`. That is exactly why A2.5.6 defines the rule on
the union of the two flags rather than on `location`: keying off the schedule's location string alone
would have handed Jacksonville a home-field bonus for a game in London.

The design matrix has no missing cells to worry about here: any nulls are handled by the *fitted*
imputer inside the bundle, using medians learned on 2002-2025. Imputing here with 2026 statistics
would be a small leak of predict-season information into the predict-season features.

### Section 4 test

Shape first. 272 games is 32 teams × 17 games ÷ 2, and any other number means the schedule
snapshot is partial. Then the two structural facts that would corrupt the output silently: the venue
join must not duplicate or drop rows, and every team must appear exactly 17 times across home and
away. The last assertion re-confirms no 2026 outcome exists in the panel.

In [ ]:
if RUN_TESTS:
    assert len(games) == len(sched), "the venue join is not 1:1"
    assert len(TEAMS) == 32, f"{len(TEAMS)} teams in the {PREDICT_SEASON} schedule"
    assert len(g26) == 272, f"{len(g26)} games - expected 272"
    _appearances = (pd.concat([g26["home_franchise"], g26["away_franchise"]])
                    .value_counts())
    assert set(_appearances.unique()) == {17}, f"uneven schedule: {_appearances.value_counts().to_dict()}"
    assert list(X26.columns) == FEATURES, "design matrix column order drifted"
    assert g26["hfa_mult"].isin([0.0, 1.0]).all()
    assert panel[panel["season"] == PREDICT_SEASON][TARGET].isna().all(), \
        "the predict season has a target - it would not be a forecast"
    print(f"✓ Section 4 tests passed | {len(g26)} games, 32 teams x 17 appearances each, "
          f"venue join 1:1, {neutral} games without home field, no {PREDICT_SEASON} target")

Passed. The 2026 design is complete and balanced, and nothing in it encodes an outcome.

## Section 5. Simulate 20,000 seasons

Each simulation draws a margin for all 272 games from `N(mu, sigma)`, plus a per-team-season shock
`N(0, tau)` that is added to that team's games and subtracted from its opponents'. That is the Amendment 3
correction that stopped the raw M4 bands being too narrow. Margins inside the tie threshold score
half a win each side.

The result is a 20,000 × 32 matrix of win counts. The projection is its mean; the distribution is
its quantiles. Because the shock is antisymmetric within a game, **league wins are conserved in
every single simulation**: the 32 teams always sum to 272.

In [ ]:
SIM = eng.simulate_wins(FIT, g26, X26, n_sims=N_SIMS, seed=SEED, tau=TAU)

proj = pd.DataFrame({
    "proj_wins": SIM.mean(),
    "p10": SIM.quantile(.10), "p25": SIM.quantile(.25), "p50": SIM.quantile(.50),
    "p75": SIM.quantile(.75), "p90": SIM.quantile(.90),
    "sd": SIM.std(ddof=1),
}).sort_values("proj_wins", ascending=False)

sums = SIM.to_numpy().sum(axis=1)
print(f"simulations     : {N_SIMS:,}  seed {SEED}  tau {TAU:g}")
print(f"conservation    : every simulation sums to {sums.min():.0f}"
      f"{'' if sums.min() == sums.max() else f'-{sums.max():.0f}'} "
      f"(scheduled games = {len(g26)})")
print(f"projected range : {proj['proj_wins'].min():.2f} to {proj['proj_wins'].max():.2f} wins")
print(f"mean band width : {(proj['p90'] - proj['p10']).mean():.2f} wins (p10 to p90)")
print()
print(pd.concat([proj.head(6), proj.tail(6)]).to_string(float_format="%.2f"))

Every simulation sums to exactly 272, so the projections are internally consistent as a league. No
team can be talked up without another coming down.

The spread of projected wins is narrow relative to what actually happens in a season, and the p10-p90
band is wide, 7.4 wins on average. That is the honest shape of this problem: preseason team-strength
information explains a modest share of season win variance, and a calibrated model says so with a wide
interval rather than a confident point. This is the same model that lands 0.12 wins further from the
truth than the archived consensus, so read the ordering as a projection, not a ranking anyone should
act on.

### Section 5 test

Conservation is the load-bearing invariant, so it is asserted on **every** simulation, not on the
means, because a bug in the shock could conserve on average while violating each draw. Then the
usual distribution sanity: quantiles ordered, wins inside `[0, 17]`, and a reproducibility check that
re-runs the simulation at the same seed and requires the identical matrix.

In [ ]:
if RUN_TESTS:
    assert SIM.shape == (N_SIMS, 32)
    assert np.allclose(sums, len(g26)), "league wins are not conserved in every simulation"
    assert (proj["p10"] <= proj["p25"]).all() and (proj["p25"] <= proj["p50"]).all()
    assert (proj["p50"] <= proj["p75"]).all() and (proj["p75"] <= proj["p90"]).all()
    assert proj["proj_wins"].between(0, 17).all() and np.isfinite(proj.to_numpy()).all()
    assert abs(proj["proj_wins"].sum() - len(g26)) < 1e-9
    _repro = eng.simulate_wins(FIT, g26, X26, n_sims=2000, seed=SEED, tau=TAU)
    assert _repro.equals(eng.simulate_wins(FIT, g26, X26, n_sims=2000, seed=SEED, tau=TAU)), \
        "the simulation is not reproducible at a fixed seed"
    assert (proj["sd"] > 0).all()
    print(f"✓ Section 5 tests passed | conservation holds in all {N_SIMS:,} simulations, "
          f"quantiles ordered p10≤p25≤p50≤p75≤p90, projections sum to {len(g26)}, "
          f"simulation reproducible at seed {SEED}")

Passed. Conservation holds in all 20,000 draws, not on average but in each one, and the simulation is
bit-for-bit reproducible at a fixed seed.

## Section 6. Assemble the artifact frame

The final schema, pinned here. Fourteen columns: identity, the projection, five quantiles and a
standard deviation, then four provenance columns stamped on **every row** so a copied CSV can never
be separated from the model version and the claim licence that governs it.

No market columns, for the two reasons in the intro. This is a deviation from the scaffold's planned
schema, which was written before Amendment 1 closed gate C; the plan was never frozen in
`PREREGISTRATION.md` §8, which pins only the filename.

In [ ]:
COLUMNS = ["season", "team", "games_scheduled", "proj_wins",
           "p10", "p25", "p50", "p75", "p90", "sd",
           "model_family", "model_sha256", "audit_verdict", "claim_label", "generated_at"]

GENERATED_AT = datetime.now(timezone.utc).isoformat()

games_sched = (panel[panel["season"] == PREDICT_SEASON]
               .set_index("franchise")["games_scheduled"].astype(int))

out = (proj.reset_index().rename(columns={"index": "team"}))
out.insert(0, "season", PREDICT_SEASON)
out["games_scheduled"] = out["team"].map(games_sched)
out["model_family"]  = MODEL_FAMILY
out["model_sha256"]  = MODEL_HASH
out["audit_verdict"] = VERDICT
out["claim_label"]   = CLAIM_LABEL
out["generated_at"]  = GENERATED_AT
out = out[COLUMNS].sort_values("proj_wins", ascending=False).reset_index(drop=True)

for c in ["proj_wins", "p10", "p25", "p50", "p75", "p90", "sd"]:
    out[c] = out[c].round(4)

guard(out, "futures_predictions")
guard(COLUMNS, "futures_predictions.columns")

print(f"{len(out)} rows x {len(out.columns)} columns")
print(f"columns : {COLUMNS}")
print(f"stamped : {MODEL_FAMILY} / {MODEL_HASH[:16]}… / {VERDICT} / {GENERATED_AT}")
print(f"label   : {CLAIM_LABEL}")
print()
print(out.head(5).to_string(index=False))

32 rows, 15 columns (the 14 described plus `season`), scanned clean by the Tier-C guard, both the
column names and every string value. `audit_verdict` is `GO-TIER-B`, whose token *tier* is on the
banned list; it passes only because it is on the exact-literal allowlist, which is the mechanism
working as intended rather than a hole in it.

`generated_at` is a wall-clock stamp, so the CSV's own hash changes on every run. The stable pins are
the model sha256 carried in the rows and the parquet inputs.

### Section 6 test

Schema exactly as pinned, in order. The page's tests will assert against this list, so a silent
reorder here becomes a silent breakage there. Then per-row completeness and the league-conservation
check restated on the *artifact* rather than the simulation.

**Two separate mechanisms exclude market columns, and only one of them is the word guard.**
`tier_lock` matches whole tokens against a banned vocabulary, and `p_over` / `p_under` / `p_push`
tokenise to `{p, over}`, `{p, under}`, `{p, push}`, and **none of those are banned words**, so the
guard would pass them. Widening the vocabulary to `over`/`under` is not the fix: those words are
ordinary English and would false-positive on prose values. The exclusion is therefore enforced by an
**explicit column denylist**, asserted below as its own check. The red controls use vocabulary the
guard genuinely covers, so they test the guard rather than flattering it.

In [ ]:
if RUN_TESTS:
    assert list(out.columns) == COLUMNS, "artifact schema drifted from the pinned list"
    assert len(out) == 32 and out["team"].nunique() == 32
    assert out["season"].eq(PREDICT_SEASON).all()
    assert out["games_scheduled"].eq(17).all()
    assert out.notna().all().all(), "the artifact has nulls"
    assert abs(out["proj_wins"].sum() - len(g26)) < 1e-3, "league conservation lost in rounding"
    assert (out["p10"] <= out["p50"]).all() and (out["p50"] <= out["p90"]).all()
    assert out["proj_wins"].is_monotonic_decreasing
    for c in ("model_family", "model_sha256", "audit_verdict", "claim_label", "generated_at"):
        assert out[c].nunique() == 1, f"{c} is not constant across rows"
    # mechanism 1 - explicit denylist. This, NOT the word guard, is what keeps market columns out.
    MARKET_COLS = ("win_total_line", "book", "line_as_of", "price_over", "price_under",
                   "p_over", "p_under", "p_push")
    assert not any(t in COLUMNS for t in MARKET_COLS), \
        "a market column is present while gate C is shut"
    # ...and the denylist is load-bearing precisely because the guard does NOT cover these names:
    from tier_lock import TIER_C_BANNED, tokens
    _uncovered = [c for c in MARKET_COLS if not (tokens(c) & TIER_C_BANNED)]
    assert "p_over" in _uncovered, "tier_lock now covers p_over - re-derive which mechanism applies"
    # mechanism 2 - the word guard, proven live on vocabulary it does cover
    for _red in (pd.DataFrame({"ev_estimate": [0.5]}),
                 pd.DataFrame({"note": ["strong value play"]}),
                 pd.DataFrame({"confidence": ["HIGH"]})):
        try:
            guard(_red, "red")
            raise AssertionError(f"red control not rejected: {list(_red.columns)}")
        except TierCViolation:
            pass
    print(f"✓ Section 6 tests passed | schema matches the pinned {len(COLUMNS)} columns in order, "
          f"32 complete rows summing to {out['proj_wins'].sum():.0f} wins, provenance constant, "
          f"{len(MARKET_COLS)} market columns excluded by denylist "
          f"({len(_uncovered)} of them invisible to the word guard), 3 red controls rejected")

Passed, and it recorded something worth knowing. Of the 8 market column names on the denylist,
six (`win_total_line`, `book`, `line_as_of`, `p_over`, `p_under`, `p_push`) carry **no banned token at all**, so the
Tier-C word guard would have let them through. The guard is a backstop against forbidden *language*;
it is not a schema check, and this notebook does not treat it as one. The denylist assertion is what
keeps gate-C columns out of the artifact, and it is asserted to remain load-bearing. If `tier_lock`
ever grows to cover these names, the test fails and forces a re-read of which mechanism applies.

The word guard is separately proven live on three phrases it *does* cover, so the clean scan above is
evidence rather than silence.

## Section 7. Write the CSV and read it back

Write `futures/futures_predictions.csv`, then re-read it from disk and re-run the structural checks
against the *parsed* file. Round-tripping matters because the page never sees the in-memory frame.
It sees whatever pandas parses back, with whatever dtypes CSV inference gives it.

In [ ]:
if WRITE_ARTIFACTS:
    OUT.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(OUT, index=False)

back = pd.read_csv(OUT) if OUT.exists() else None
OUT_HASH = sha256_file(OUT) if OUT.exists() else None

if back is not None:
    print(f"wrote      : {_rel(OUT)}  ({OUT.stat().st_size:,} bytes)")
    print(f"sha256     : {OUT_HASH}")
    print(f"read back  : {back.shape[0]} rows x {back.shape[1]} columns")
    print(f"dtypes ok  : {back['proj_wins'].dtype} / {back['team'].dtype}")
    print()
    print(back.head(10).to_string(index=False,
          columns=["season", "team", "proj_wins", "p10", "p50", "p90"]))
else:
    print("WRITE_ARTIFACTS is False - nothing written")

The artifact is on disk and parses back to 32 rows with the numeric columns typed numerically. Its
sha256 is printed for the record, but note it is **not** a stable pin: `generated_at` changes it on
every run. The reproducible identity of this artifact is the model hash carried inside it.

### Section 7 test

Everything asserted on the in-memory frame, re-asserted on the parsed file: schema, row count,
conservation, quantile ordering, and a fresh Tier-C scan of the *round-tripped* strings. The final
assertion confirms the runtime separation §8 requires: the page reads this file with pandas alone,
so a number that needs the model to interpret it would be a design failure.

In [ ]:
if RUN_TESTS and WRITE_ARTIFACTS:
    assert back is not None and list(back.columns) == COLUMNS
    assert len(back) == 32 and back["team"].nunique() == 32
    assert back.notna().all().all()
    assert abs(back["proj_wins"].sum() - 272) < 1e-3
    assert (back["p10"] <= back["p50"]).all() and (back["p50"] <= back["p90"]).all()
    assert np.allclose(back["proj_wins"].to_numpy(), out["proj_wins"].to_numpy())
    assert back["model_sha256"].iloc[0] == meta["model"]["sha256"]
    assert back["audit_verdict"].iloc[0] == VERDICT
    assert "does not beat" in back["claim_label"].iloc[0]
    guard(back, "futures_predictions.csv")
    for c in ("proj_wins", "p10", "p25", "p50", "p75", "p90", "sd"):
        assert pd.api.types.is_numeric_dtype(back[c]), f"{c} did not parse as numeric"
    print(f"✓ Section 7 tests passed | {_rel(OUT)} round-trips to {len(back)} rows x "
          f"{len(back.columns)} columns, conservation and ordering survive the CSV, "
          f"numeric dtypes parse, Tier-C scan clean on the parsed file")

Passed. The file on disk is the file the page will read, and it holds every invariant the in-memory
frame did.

## Conclusion and next steps

**Written:** `futures/futures_predictions.csv`, 32 rows, 15 columns, 2026 projected wins plus a
five-quantile win distribution from 20,000 simulated seasons of the frozen M4-c model.

**What this notebook did and did not do.** It loaded `win_totals_model.pkl`, verified its hash
against `04`'s record, applied it to the published 2026 schedule, and wrote the result. It fitted
nothing, selected nothing, and read every model constant out of the bundle. The model's training
window ends at 2025.

**Findings worth carrying forward:**

* League wins are conserved in **every** simulation, not on average. The 32 projections sum to
  exactly 272.
* The p10-p90 band averages 7.4 wins. That width is the calibrated answer, produced under
  Amendment 3 after raw M4's 80% interval covered only 65% of outcomes.
* The market columns the scaffold planned are **absent**, and this is a deliberate deviation:
  `tier_c_open` is `false`, and no 2026 archived line exists to attach even if it were open.

**The label that must travel with this artifact anywhere it is displayed:**
*does not beat the archived market consensus; BACKTESTED, NOT LIVE-VALIDATED.* It is stamped into
every row as `claim_label` so it cannot be separated from the numbers.

**Limitations restated, not buried.** M4-c is 0.12 wins *further* from the realized win count than
the archived consensus over the 10-fold backtest (§7 gate B failed for every model tested). The
production `tau` sits at the top of its frozen grid with inner coverage still below the 0.80 target,
so the true optimum probably lies above the grid. A3.4 forbids widening it without a new amendment.
And the benchmark itself is an archived consensus of unattributed sportsbook origin with
date-granularity timing, never a named book.

**Next steps, outside this notebook:**

1. Build `site_pages/page_futures.py` reading this CSV with pandas and Streamlit only: no model,
   no simulator, no training dependency in the deployed runtime (§8 runtime separation).
2. Register a "Season Totals" entry in `nav_registry.PAGES` and add the page to the offline AppTest
   sweep, checked on **mobile and desktop** per the standing site rule.
3. The page must render the projection and the distribution, state the gate B result plainly, and
   carry the claim label. No side, no probability against a line, no confidence tier.
4. Re-run this notebook when the 2026 schedule or the panel changes; it is cheap and fully
   deterministic at a fixed seed.